In [1]:
! git clone https://github.com/George-Malak/Aegis-AI-Violence-Detection-System.git

fatal: destination path 'Aegis-AI-Violence-Detection-System' already exists and is not an empty directory.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import sys
import os
from pathlib import Path
import torch.nn as nn
import torch
import cv2
from torch.utils.data import DataLoader



SRC_PATH = Path('/content/Aegis-AI-Violence-Detection-System/ml_core').resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from src.preprocessing.video_preprocessor import VideoPreprocessor
from src.data.dataset_loader import DatasetLoader, OnTheFlyVideoDataset
from src.models.videomae_model import VideoPreprocessor, VideoMAEClassifier
from configs.MLFlow import  MLflowLogger

# Load Data

In [4]:
ROOT_DIR = Path('/content/drive/MyDrive/DEPI Project').resolve()

train_finder = DatasetLoader(drive_root=f"{ROOT_DIR}/Train")
train_manifest = train_finder.load_dataset()

val_finder = DatasetLoader(drive_root=f"{ROOT_DIR}/Val")
val_manifest = val_finder.load_dataset()

test_finder = DatasetLoader(drive_root=f"{ROOT_DIR}/Test")
test_manifest = test_finder.load_dataset()

print(f"LOADED DONE !")
print(f"Train Vid : {len(train_manifest)}")
print(f"Validation Vid :   {len(val_manifest)}")
print(f"Test Vid:  {len(test_manifest)}")


LOADED DONE !
Train Vid : 5464
Validation Vid :   145
Test Vid:  166


In [5]:
test_manifest[5]

{'class_name': 'Shoplifting',
 'video_path': '/content/drive/.shortcut-targets-by-id/1dY6DTYepRVAzSHcM_vTLO7qysH-tzoji/DEPI Project/Test/Shoplifting/Shoplifting028_x264.mp4',
 'file_name': 'Shoplifting028_x264.mp4'}

In [6]:
unique_classes = sorted(list(set(sample['class_name'] for sample in train_manifest)))
class_to_idx = {name: idx for idx, name in enumerate(unique_classes)}
print(f"Class Mapping indices discovered: {class_to_idx}")

Class Mapping indices discovered: {'Abuse': 0, 'Arrest': 1, 'Arson': 2, 'Assault': 3, 'Burglary': 4, 'Explosion': 5, 'Fighting': 6, 'Normal': 7, 'RoadAccidents': 8, 'Robbery': 9, 'Shooting': 10, 'Shoplifting': 11, 'Stealing': 12, 'Vandalism': 13}


In [16]:
preprocessor = VideoPreprocessor(target_frames=16, target_size=(224, 224))

# Train
train_dataset = OnTheFlyVideoDataset(train_manifest, preprocessor, class_to_idx)
val_dataset = OnTheFlyVideoDataset(val_manifest, preprocessor, class_to_idx)
# val
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

# MLFlow
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
logger = MLflowLogger(experiment_name="Violence_Detection_Experiments")

# Model VideoMEA

In [17]:
logger.start_run(run_name="VideoMAE_Run_1")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VideoMAEClassifier(num_classes=len(unique_classes)).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
logger.log_parameters({"lr": 2e-5, "epochs": 3, "optimizer": "AdamW"})

MLflow Run Started: 'VideoMAE_Run_1' under Experiment tracking.
Loading pretrained VideoMAE backbone: MCG-NJU/videomae-base...


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED | 
encoder_to_decoder.weight                                            | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.v_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.q_bias       | UNEXPECTED

In [ ]:
epochs = 3
print("\nInitiating training runtime execution sequence...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (videos, labels) in enumerate(train_loader):
        videos, labels = videos.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(videos)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if (batch_idx + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{batch_idx+1}/{len(train_loader)}], Batch Loss: {loss.item():.4f}")

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for videos, labels in val_loader:
            videos, labels = videos.to(device), labels.to(device)
            logits = model(videos)
            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    avg_train_loss = running_loss / len(train_loader)
    logger.log_epoch_metrics(
        metrics={"train_loss": avg_train_loss, "val_accuracy": val_correct/val_total},
        epoch=epoch
    )
    print(f"--> Epoch [{epoch+1}/{epochs}] Done. Avg Train Loss: {running_loss/len(train_loader):.4f} | Val Accuracy: {val_correct/val_total:.4f}\n")
logger.log_model_checkpoint(model=model)
logger.end_run()


Initiating training runtime execution sequence...
Epoch [1/3], Step [50/1366], Batch Loss: 2.8879
Epoch [1/3], Step [100/1366], Batch Loss: 2.0545
Epoch [1/3], Step [150/1366], Batch Loss: 2.2719
Epoch [1/3], Step [200/1366], Batch Loss: 1.6999
Epoch [1/3], Step [250/1366], Batch Loss: 1.1356
Epoch [1/3], Step [300/1366], Batch Loss: 2.0260
Epoch [1/3], Step [350/1366], Batch Loss: 1.8463
Epoch [1/3], Step [400/1366], Batch Loss: 1.5114
Epoch [1/3], Step [450/1366], Batch Loss: 2.0076
Epoch [1/3], Step [500/1366], Batch Loss: 1.6077
Epoch [1/3], Step [550/1366], Batch Loss: 1.3824
Epoch [1/3], Step [600/1366], Batch Loss: 0.8568
Epoch [1/3], Step [650/1366], Batch Loss: 1.7885
Epoch [1/3], Step [700/1366], Batch Loss: 1.3710
Epoch [1/3], Step [750/1366], Batch Loss: 0.5826
Epoch [1/3], Step [800/1366], Batch Loss: 1.2766
Epoch [1/3], Step [850/1366], Batch Loss: 1.6055
Epoch [1/3], Step [900/1366], Batch Loss: 0.7433
Epoch [1/3], Step [950/1366], Batch Loss: 0.8612
Epoch [1/3], Step [

In [ ]:
def predict_single_video(video_path, model, preprocessor, class_to_idx, device):
    """Extracts features from an unseeen single path file and prints the predicted crime class."""
    model.eval()
    tensor = preprocessor.preprocess(video_path)
    if tensor is None:
        return "Error reading video file structure."

    tensor = torch.from_numpy(tensor).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)
        pred_idx = torch.argmax(logits, dim=1).item()

    idx_to_class = {v: k for k, v in class_to_idx.items()}
    return idx_to_class[pred_idx]

sample_test_video = test_manifest[0]["video_path"]
predicted_crime = predict_single_video(sample_test_video, model, preprocessor, class_to_idx, device)
print(f"Target Video Sample : {sample_test_video}")
print(f"Aegis-AI Prediction : {predicted_crime}")